# Week 5 — Shallow Neural Network Classifier

## Objective

This notebook extends the Week 3 CSP+LDA and Week 4 classical machine-learning benchmarks by implementing a shallow feedforward neural network in PyTorch for robot sensor mode classification.

The main objectives are to:

1. Train a shallow neural network using the same PATROL vs. FAULT classification task and 80/20 train-test split used in Week 3.
2. Evaluate training loss and validation accuracy across epochs.
3. Perform a small hyperparameter sensitivity analysis over hidden-layer size, dropout rate, and learning rate.
4. Evaluate the best neural-network configuration using accuracy, precision, recall, and F1 score.
5. Compare the neural network with the Week 3 CSP+LDA model and the best Week 4 classical classifier.
6. Use repeated seeded runs and a paired t-test to determine whether differences between the top-performing methods are statistically significant.
7. Assess whether any improvement from the neural network justifies its additional complexity for edge deployment.

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import ttest_rel

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

PyTorch version: 2.13.0
NumPy version: 2.4.6
Pandas version: 3.0.5


In [2]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Random seed:", SEED)
print("Device:", device)

Random seed: 42
Device: cpu


## 1. Data Loading and Week 3-Compatible Preprocessing

To maintain a fair comparison with the Week 3 CSP+LDA benchmark, this notebook uses the same high-frequency robot sensor dataset and PATROL vs. FAULT classification task.

The high-frequency sensor stream is segmented into 30-second windows at a sampling rate of 20 Hz, corresponding to 600 samples per window.

The same sensor channels used in the Week 3 pipeline are retained:

- motor current
- x-axis acceleration
- y-axis acceleration
- z-axis acceleration
- IMU magnitude

The train-test split is reproduced using an 80/20 stratified split with random state 42.

In [3]:
DATA_PATH = "../data/robot_sensor_highfreq.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (48000, 8)


,time,mode,motor_current,accel_x,accel_y,accel_z,imu_magnitude,window_id
0,0.00,PATROL,2.049671,0.060559,0.010018,9.924071,9.924261,0
1,0.05,PATROL,2.050013,-0.042440,-0.003066,9.825160,9.825252,0
2,0.10,PATROL,2.189131,0.131741,0.071587,9.804799,9.805945,0
3,0.15,PATROL,2.330852,0.200482,0.134262,9.774396,9.777374,0
4,0.20,PATROL,2.200534,0.153513,0.121466,9.774075,9.776035,0


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset info:")
df.info()

print("\nMode distribution:")
print(df["mode"].value_counts())

Columns:
['time', 'mode', 'motor_current', 'accel_x', 'accel_y', 'accel_z', 'imu_magnitude', 'window_id']

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   time           48000 non-null  float64
 1   mode           48000 non-null  str    
 2   motor_current  48000 non-null  float64
 3   accel_x        48000 non-null  float64
 4   accel_y        48000 non-null  float64
 5   accel_z        48000 non-null  float64
 6   imu_magnitude  48000 non-null  float64
 7   window_id      48000 non-null  int64  
dtypes: float64(6), int64(1), str(1)
memory usage: 2.9 MB

Mode distribution:
mode
PATROL      12000
ALERT       12000
CHARGING    12000
FAULT       12000
Name: count, dtype: int64


In [5]:
# Keep the same binary classification task as Week 3
binary_df = df[df["mode"].isin(["PATROL", "FAULT"])].copy()

SENSOR_COLS = [
    "motor_current",
    "accel_x",
    "accel_y",
    "accel_z",
    "imu_magnitude"
]

print("Binary dataset shape:", binary_df.shape)
print("\nMode distribution:")
print(binary_df["mode"].value_counts())

print("\nNumber of unique windows:")
print(binary_df["window_id"].nunique())

Binary dataset shape: (24000, 8)

Mode distribution:
mode
PATROL    12000
FAULT     12000
Name: count, dtype: int64

Number of unique windows:
20


### Window-Level Feature Representation

The high-frequency dataset contains multiple sensor samples within each 30-second operational window. To construct fixed-length inputs for a feedforward neural network, each window is represented using summary statistics from the five sensor channels.

For every sensor channel, the following statistics are extracted:

- mean
- standard deviation
- minimum
- maximum
- root mean square (RMS)

With five sensor channels and five statistics per channel, each 30-second window is represented by a 25-dimensional feature vector.

This preserves the same window-level classification structure used in the Week 3 benchmark while providing fixed-dimensional inputs suitable for a shallow feedforward neural network.

In [6]:
def rms(x):
    return np.sqrt(np.mean(np.square(x)))


feature_rows = []

for window_id, group in binary_df.groupby("window_id"):
    
    row = {
        "window_id": window_id,
        "mode": group["mode"].iloc[0]
    }
    
    for col in SENSOR_COLS:
        values = group[col].values
        
        row[f"{col}_mean"] = np.mean(values)
        row[f"{col}_std"] = np.std(values)
        row[f"{col}_min"] = np.min(values)
        row[f"{col}_max"] = np.max(values)
        row[f"{col}_rms"] = rms(values)
    
    feature_rows.append(row)


features_df = pd.DataFrame(feature_rows)

print("Window-level dataset shape:", features_df.shape)
print("\nClass distribution:")
print(features_df["mode"].value_counts())

display(features_df.head())

Window-level dataset shape: (20, 27)

Class distribution:
mode
PATROL    20
Name: count, dtype: int64


,window_id,mode,motor_current_mean,motor_current_std,motor_current_min,motor_current_max,motor_current_rms,accel_x_mean,accel_x_std,accel_x_min,...,accel_z_mean,accel_z_std,accel_z_min,accel_z_max,accel_z_rms,imu_magnitude_mean,imu_magnitude_std,imu_magnitude_min,imu_magnitude_max,imu_magnitude_rms
0,0,PATROL,3.246884,1.388130,1.427536,6.394661,3.531170,-0.001174,0.426919,-1.396704,...,9.812881,0.342102,8.761679,10.832680,9.818842,9.829281,0.341670,8.767102,10.878254,9.835217
1,1,PATROL,3.247494,1.382599,1.344306,6.697801,3.529561,0.003341,0.419453,-1.512459,...,9.813205,0.352025,8.579672,10.813154,9.819517,9.829438,0.351520,8.580255,10.818737,9.835722
2,2,PATROL,3.263389,1.403481,1.387518,6.807228,3.552389,0.001577,0.421768,-1.372750,...,9.811447,0.345387,8.718393,10.851637,9.817524,9.827461,0.345023,8.738981,10.858304,9.833516
3,3,PATROL,3.249959,1.390097,1.356082,6.371952,3.534771,0.006125,0.427081,-1.304555,...,9.804113,0.341890,8.765072,10.933663,9.810072,9.820531,0.341552,8.823486,10.959201,9.826468
4,4,PATROL,3.258319,1.397658,1.458352,6.793703,3.545432,-0.000855,0.428744,-1.392858,...,9.810650,0.338558,8.681198,10.809768,9.816490,9.827382,0.338079,8.698998,10.849405,9.833196


In [7]:
feature_cols = [
    col for col in features_df.columns
    if col not in ["window_id", "mode"]
]

X = features_df[feature_cols].values.astype(np.float32)

# PATROL = 0, FAULT = 1
y = (features_df["mode"] == "FAULT").astype(np.int64).values

window_ids = features_df["window_id"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", len(feature_cols))

print("\nClass counts:")
print("PATROL:", np.sum(y == 0))
print("FAULT :", np.sum(y == 1))

X shape: (20, 25)
y shape: (20,)
Number of features: 25

Class counts:
PATROL: 20
FAULT : 0


In [8]:
X_train_full, X_test, y_train_full, y_test, id_train_full, id_test = train_test_split(
    X,
    y,
    window_ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Full training set:", X_train_full.shape)
print("Held-out test set:", X_test.shape)

print("\nTraining class counts:")
print("PATROL:", np.sum(y_train_full == 0))
print("FAULT :", np.sum(y_train_full == 1))

print("\nTest class counts:")
print("PATROL:", np.sum(y_test == 0))
print("FAULT :", np.sum(y_test == 1))

print("\nHeld-out test window IDs:")
print(id_test)

Full training set: (16, 25)
Held-out test set: (4, 25)

Training class counts:
PATROL: 16
FAULT : 0

Test class counts:
PATROL: 4
FAULT : 0

Held-out test window IDs:
[ 7  6 14 10]


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

print("NN training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Held-out test set:", X_test.shape)

print("\nNN training class counts:")
print("PATROL:", np.sum(y_train == 0))
print("FAULT :", np.sum(y_train == 1))

print("\nValidation class counts:")
print("PATROL:", np.sum(y_val == 0))
print("FAULT :", np.sum(y_val == 1))

NN training set: (12, 25)
Validation set: (4, 25)
Held-out test set: (4, 25)

NN training class counts:
PATROL: 12
FAULT : 0

Validation class counts:
PATROL: 4
FAULT : 0


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

print("\nTraining feature mean after scaling:",
      np.round(X_train_scaled.mean(axis=0)[:5], 3))

print("Training feature std after scaling:",
      np.round(X_train_scaled.std(axis=0)[:5], 3))

Scaled training shape: (12, 25)
Scaled validation shape: (4, 25)
Scaled test shape: (4, 25)

Training feature mean after scaling: [-0. -0.  0.  0. -0.]
Training feature std after scaling: [1. 1. 1. 1. 1.]
